# Module 9: Matrix Calculus & Automatic Differentiation

Training neural networks requires computing gradients of scalar loss functions with respect to millions of matrix/vector parameters. Matrix calculus provides the notation and rules; automatic differentiation provides the efficient algorithms.

## Contents
1. Scalar, Vector, and Matrix Derivatives — Layout Conventions
2. Common Matrix Calculus Identities
3. The Chain Rule for Matrices
4. Jacobian-Vector Products (JVPs) and Vector-Jacobian Products (VJPs)
5. Forward-Mode Automatic Differentiation
6. Reverse-Mode Automatic Differentiation (Backpropagation)
7. Computational Graphs

## 1. Scalar, Vector, and Matrix Derivatives — Layout Conventions

Matrix calculus extends single-variable calculus to vectors and matrices. The key objects are:

| Numerator \ Denominator | Scalar $x$ | Vector $\mathbf{x} \in \mathbb{R}^n$ | Matrix $X \in \mathbb{R}^{m \times n}$ |
|---|---|---|---|
| Scalar $y$ | $\frac{\partial y}{\partial x}$ (scalar) | $\frac{\partial y}{\partial \mathbf{x}}$ (gradient, $n$-vector) | $\frac{\partial y}{\partial X}$ ($m \times n$ matrix) |
| Vector $\mathbf{y} \in \mathbb{R}^m$ | $\frac{\partial \mathbf{y}}{\partial x}$ ($m$-vector) | $\frac{\partial \mathbf{y}}{\partial \mathbf{x}}$ (Jacobian, $m \times n$) | — |

We use the **numerator layout** (Jacobian) convention throughout, which is the standard in ML.

### The Gradient
For a scalar function $f: \mathbb{R}^n \to \mathbb{R}$, the gradient is:
$$\nabla_\mathbf{x} f = \frac{\partial f}{\partial \mathbf{x}} = \begin{bmatrix} \frac{\partial f}{\partial x_1} \\ \vdots \\ \frac{\partial f}{\partial x_n} \end{bmatrix}$$

### The Jacobian
For a vector function $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$, the Jacobian is the $m \times n$ matrix:
$$J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} = \begin{bmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n} \end{bmatrix}$$

In [ ]:
import numpy as np
import sympy as sp

# Compute Jacobian symbolically
x1, x2 = sp.symbols('x1 x2')
f1 = x1**2 + x2
f2 = sp.sin(x1) * x2
f = sp.Matrix([f1, f2])
x = sp.Matrix([x1, x2])

J = f.jacobian(x)
print("Jacobian:")
sp.pprint(J)

## 2. Common Matrix Calculus Identities

These are the identities you will use most frequently in deep learning:

| Expression | Derivative |
|---|---|
| $f = \mathbf{a}^T \mathbf{x}$ | $\nabla_\mathbf{x} f = \mathbf{a}$ |
| $f = \mathbf{x}^T A \mathbf{x}$ | $\nabla_\mathbf{x} f = (A + A^T) \mathbf{x}$ |
| $f = \mathbf{x}^T \mathbf{x} = \|\mathbf{x}\|^2$ | $\nabla_\mathbf{x} f = 2\mathbf{x}$ |
| $\mathbf{y} = A\mathbf{x}$ | $\frac{\partial \mathbf{y}}{\partial \mathbf{x}} = A$ |
| $f = \text{tr}(AB)$ | $\frac{\partial f}{\partial A} = B^T$ |
| $f = \log \det(X)$ | $\frac{\partial f}{\partial X} = X^{-T}$ |

In [ ]:
# Verify: d/dx (x^T A x) = (A + A^T) x
x1, x2 = sp.symbols('x1 x2')
A = sp.Matrix([[2, 1], [3, 4]])
x = sp.Matrix([x1, x2])
f = (x.T @ A @ x)[0, 0]  # scalar

grad_computed = sp.Matrix([sp.diff(f, xi) for xi in [x1, x2]])
grad_formula = (A + A.T) @ x

print("Computed gradient:", grad_computed.T)
print("Formula gradient: ", grad_formula.T)
print("Match:", sp.simplify(grad_computed - grad_formula) == sp.zeros(2, 1))

## 3. The Chain Rule for Matrices

For composed functions $\mathbf{y} = f(g(\mathbf{x}))$, the chain rule in matrix form is:
$$\frac{\partial \mathbf{y}}{\partial \mathbf{x}} = \frac{\partial \mathbf{y}}{\partial \mathbf{u}} \cdot \frac{\partial \mathbf{u}}{\partial \mathbf{x}}$$

where $\mathbf{u} = g(\mathbf{x})$. This is simply matrix multiplication of Jacobians.

For a scalar loss $L = L(\mathbf{y}(\mathbf{x}))$:
$$\nabla_\mathbf{x} L = J_{\mathbf{y}/\mathbf{x}}^T \nabla_\mathbf{y} L$$

This is the fundamental equation behind backpropagation.

## 4. Jacobian-Vector Products (JVPs) and Vector-Jacobian Products (VJPs)

For a function $f: \mathbb{R}^n \to \mathbb{R}^m$ with Jacobian $J \in \mathbb{R}^{m \times n}$:

- **JVP** (forward-mode): Given a tangent vector $\mathbf{v} \in \mathbb{R}^n$, compute $J \mathbf{v} \in \mathbb{R}^m$. Cost: one forward pass.
- **VJP** (reverse-mode): Given a cotangent vector $\mathbf{u} \in \mathbb{R}^m$, compute $\mathbf{u}^T J \in \mathbb{R}^n$. Cost: one backward pass.

For neural networks with scalar loss ($m=1$) and many parameters ($n \gg 1$), VJPs (reverse-mode) are vastly more efficient: one backward pass gives the full gradient, whereas forward-mode would require $n$ forward passes.

In [ ]:
# Numerical JVP and VJP demonstration
def f(x):
    return np.array([x[0]**2 + x[1], np.sin(x[0]) * x[1]])

def numerical_jacobian(f, x, eps=1e-5):
    n = len(x)
    m = len(f(x))
    J = np.zeros((m, n))
    for i in range(n):
        e = np.zeros(n)
        e[i] = eps
        J[:, i] = (f(x + e) - f(x - e)) / (2 * eps)
    return J

x0 = np.array([1.0, 2.0])
J = numerical_jacobian(f, x0)
print("Jacobian:\n", J)

v = np.array([1.0, 0.0])  # tangent vector
print("JVP (J @ v):", J @ v)

u = np.array([1.0, 1.0])  # cotangent vector
print("VJP (u^T @ J):", u @ J)

## 5. Forward-Mode Automatic Differentiation

Forward-mode AD propagates **dual numbers** (value, derivative) through the computation. For each elementary operation, we track both the primal value and the tangent (directional derivative).

A dual number is $a + b\epsilon$ where $\epsilon^2 = 0$. Then $f(a + b\epsilon) = f(a) + f'(a) b \epsilon$.

Forward-mode computes one column of the Jacobian per pass (i.e., one JVP).

In [ ]:
class DualNumber:
    def __init__(self, value, derivative=0.0):
        self.value = value
        self.derivative = derivative

    def __add__(self, other):
        if isinstance(other, (int, float)):
            return DualNumber(self.value + other, self.derivative)
        return DualNumber(self.value + other.value, self.derivative + other.derivative)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return DualNumber(self.value * other, self.derivative * other)
        return DualNumber(self.value * other.value,
                          self.value * other.derivative + self.derivative * other.value)

    def __pow__(self, n):
        return DualNumber(self.value ** n, n * self.value ** (n - 1) * self.derivative)

    def __repr__(self):
        return f"Dual({self.value}, {self.derivative})"

# f(x) = x^3 + 2x, f'(x) = 3x^2 + 2
x = DualNumber(3.0, 1.0)  # seed derivative = 1
result = x**3 + x * 2
print(f"f(3) = {result.value}, f'(3) = {result.derivative}")
print(f"Expected: f(3) = {3**3 + 2*3}, f'(3) = {3*3**2 + 2}")

## 6. Reverse-Mode Automatic Differentiation (Backpropagation)

Reverse-mode AD records a **computational graph** during the forward pass (the "tape"), then traverses it backward to accumulate gradients via the chain rule.

For a scalar loss $L$:
1. **Forward pass**: Compute $L$ and record all intermediate values.
2. **Backward pass**: Starting with $\frac{\partial L}{\partial L} = 1$, propagate gradients backward:
   $$\bar{x}_i = \frac{\partial L}{\partial x_i} = \sum_j \bar{x}_j \frac{\partial x_j}{\partial x_i}$$
   where $\bar{x}$ denotes the "adjoint" (gradient of the loss w.r.t. that variable).

One backward pass computes the **entire gradient** $\nabla_\theta L$ for all parameters $\theta$.

In [ ]:
class Value:
    """A simple reverse-mode AD engine (inspired by Karpathy's micrograd)."""
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

# Example: f = (a * b) + b, df/da = b, df/db = a + 1
a = Value(2.0)
b = Value(3.0)
c = a * b + b
c.backward()
print(f"f = {c.data}, df/da = {a.grad} (expected 3), df/db = {b.grad} (expected 3)")

## 7. Computational Graphs

A **computational graph** is a directed acyclic graph (DAG) where:
- **Nodes** represent variables (inputs, intermediates, outputs)
- **Edges** represent operations (addition, multiplication, activation functions)

For a simple neural network layer $\mathbf{y} = \sigma(W\mathbf{x} + \mathbf{b})$, the graph is:

```
x, W → matmul → z1
z1, b → add → z2  
z2 → sigma → y
```

Frameworks like PyTorch and JAX build these graphs dynamically (eager mode) or statically (XLA compilation) and use reverse-mode AD to compute gradients efficiently.